In [ ]:
from gams import transfer as gt
import pandas as pd
import numpy as np
sys.path.insert(0, snakemake.input.data2dd)

from data2dd_funcs import wrapdd

# Existing capacity in current year = 
# Installed cap in prev year (invested in the prev year)
# - decommissioned cap between prev and current year
# + commissioned cap between prev and current year?

# commissioned cap between prev and current year:
# should the commissioned cap between years be consider as existing in the next year?
# For now, not commissioned cap is considered

In [ ]:
# inputs
results_gdx_path = snakemake.input.modelresults
gen_comm_decomm_path = snakemake.input.gen_comm_decomm_path
store_comm_decomm_path = snakemake.input.store_comm_decomm_path
gen_dd_path = snakemake.input.gen_dd_path
store_dd_path = snakemake.input.store_dd_path

# extra params
model_year = int(snakemake.wildcards.model_year)
model_years = snakemake.params.model_years

# outputs # missing something to not load all the data?
m = gt.Container(results_gdx_path, system_directory=snakemake.params.gamspath)


idx_prev_year = model_years.index(model_year)-1
prev_year = model_years[idx_prev_year]

In [ ]:
## Generation


In [ ]:
# Load previous capacities

# region
cap0_gen_exist_r = m["var_exist_vre_pcap_r"].records
cap0_gen_new_r = m["var_new_vre_pcap_r"].records

cap0_gen_exist_r = (
    cap0_gen_exist_r
    .assign(lt = "FX")
    .set_index(["z","r","vre","lt"])
    [["level"]]
)

cap0_gen_new_r = (
    cap0_gen_new_r
    .assign(lt = "FX")
    .set_index(["z","r","vre","lt"])
    [["level"]]
)

cap0_gen_r = cap0_gen_new_r+cap0_gen_exist_r

# zone
cap0_gen_z = m["var_tot_pcap_z"].records
ecap0_gen_z = m["var_tot_hydro_ecap_z"].records

cap0_gen_z = (
    cap0_gen_z
    .assign(lt = "FX")
    .set_index(["z","g","lt"])
    [["level"]]
)

ecap0_gen_z = (
    ecap0_gen_z
    .assign(lt = "FX")
    .rename(columns={"hydro_res":"g"})
    .set_index(["z","g","lt"])
    [["level"]]
)

In [ ]:
# Load commissioning and decommissioning plans
gen_comm_decomm_cap_full = pd.read_csv(gen_comm_decomm_path)

vre = list(m["vre"].records.g)
g = vre = list(m["g"].records.uni)

# Fix later: hardcoded - The original files should contain the correct names
map_tech = {
    "nuclear": "NuclearEPR",
    "solar": "Solar",
    "onshore": "Windonshore",
    "offshore": "Windoffshore",
}

gen_comm_decomm_cap = (
    gen_comm_decomm_cap_full
    .query("Year<=@model_year")
    .query("Year>@prev_year")
    .drop(columns=["Year"])
)

gen_commissioned_cap = (
    gen_comm_decomm_cap
    .query("type=='commisioning'")
    .drop(columns=["type"])
)
gen_decommissioned_cap = (
    gen_comm_decomm_cap
    .query("type=='decommisioning'")
    .drop(columns=["type"])
)


# region
# Only vre 
commissioned_cap_r = (
    gen_commissioned_cap
    .replace(map_tech)
    .query("Technology in @vre")
    .rename(columns={"Technology":"vre","zone":"z","region":"r","value":"level"})
    .assign(lt="LO")
    .set_index(["z","r","vre","lt"])
    .div(1000)
)

decommissioned_cap_r = (
    gen_decommissioned_cap
    .replace(map_tech)
    .query("Technology in @vre")
    .rename(columns={"Technology":"vre","zone":"z","region":"r","value":"level"})
    .assign(lt="FX")
    .set_index(["z","r","vre","lt"])
    .div(1000)
)

# zone
# only not vre but g
commissioned_cap_z = (
    gen_commissioned_cap
    .replace(map_tech)
    .query("Technology in @g")
    .query("Technology not in @vre")
    .drop(columns=["region"])
    .groupby(["Technology","zone"]).sum()
    .reset_index()
    .rename(columns={"Technology":"g","zone":"z","value":"level"})
    .assign(lt="LO")
    .set_index(["z","g","lt"])
    .div(1000)
)

decommissioned_cap_z = (
    gen_decommissioned_cap
    .replace(map_tech)
    .query("Technology in @g")
    .query("Technology not in @vre")
    .drop(columns=["region"])
    .groupby(["Technology","zone"]).sum()
    .reset_index()
    .rename(columns={"Technology":"g","zone":"z","value":"level"})
    .assign(lt="FX")
    .set_index(["z","g","lt"])
    .div(1000)
)



In [ ]:
# cap for the next period is:
# tot cap from prev - decommissioned between prev and now
# + commissioned between prev and now?
cap_gen_r = (
    cap0_gen_r
    .sub(decommissioned_cap_r, fill_value=0)
    # .add(commissioned_cap_r, fill_value=0)
    .reset_index()
)

cap_gen_z = (
    cap0_gen_z
    .sub(decommissioned_cap_z, fill_value=0)
    # .add(commissioned_cap_r, fill_value=0)
    .reset_index()
)

ecap_gen_z = (
    ecap0_gen_z
    .reset_index()
)

In [ ]:
cap_gen_z_agg = pd.DataFrame(cap_gen_z["z"].astype(str) +"."+ cap_gen_z["g"].astype(str) +"."+ cap_gen_z["lt"].astype(str),columns=["gen_exist_pcap_z"])
cap_gen_z_agg["level"] = cap_gen_z[["level"]]

cap_gen_r_agg = pd.DataFrame(cap_gen_r["vre"].astype(str) +"."+ cap_gen_r["z"].astype(str) +"."+ cap_gen_r["r"].astype(str) +"."+ cap_gen_r["lt"].astype(str),columns=["gen_exist_pcap_r"])
cap_gen_r_agg["level"] = cap_gen_r[["level"]]

ecap_gen_z_agg = pd.DataFrame(ecap_gen_z["z"].astype(str) +"."+ ecap_gen_z["g"].astype(str) +"."+ ecap_gen_z["lt"].astype(str),columns=["gen_exist_ecap_z"])
ecap_gen_z_agg["level"] = ecap_gen_z[["level"]]


In [ ]:
## Commissioned

# comm_cap_z = (
#     commissioned_cap_z
#     .reset_index()
# )

# comm_gen_z_agg = pd.DataFrame(comm_cap_z["z"].astype(str) +"."+ comm_cap_z["g"].astype(str) +"."+ comm_cap_z["lt"].astype(str),columns=["gen_comm_pcap_z"])
# comm_gen_z_agg["level"] = comm_cap_z[["level"]]


# comm_gen_z_out = pd.DataFrame(wrapdd(comm_gen_z_agg.values,comm_gen_z_agg.columns[0],"parameter"))
# comm_gen_z_out["out"] = comm_gen_z_out[0]+" "+comm_gen_z_out[1].astype(str)


In [ ]:
## Add existing cap to gen dd file
gen_dd = pd.read_csv(gen_dd_path,skip_blank_lines=False,header=None)

cap_gen_z_out = pd.DataFrame(wrapdd(cap_gen_z_agg.values,cap_gen_z_agg.columns[0],"parameter"))
cap_gen_z_out["out"] = cap_gen_z_out[0]+" "+cap_gen_z_out[1].astype(str)

cap_gen_r_out = pd.DataFrame(wrapdd(cap_gen_r_agg.values,cap_gen_r_agg.columns[0],"parameter"))
cap_gen_r_out["out"] = cap_gen_r_out[0]+" "+cap_gen_r_out[1].astype(str)

ecap_gen_z_out = pd.DataFrame(wrapdd(ecap_gen_z_agg.values,ecap_gen_z_agg.columns[0],"parameter"))
ecap_gen_z_out["out"] = ecap_gen_z_out[0]+" "+ecap_gen_z_out[1].astype(str)

gen_dd_out = np.concatenate(
        (gen_dd.values,
        cap_gen_z_out.loc[:,["out"]].values,
        ecap_gen_z_out.loc[:,["out"]].values,
        cap_gen_r_out.loc[:,["out"]].values,
        ),
        axis=0)
np.savetxt(
            gen_dd_path, gen_dd_out, delimiter=" ", fmt="%s"
        )


In [ ]:
## Storage

In [ ]:
# Load prev storage cap

cap0_store_z = m["var_tot_store_pcap_z"].records
ecap0_store_z = m["var_tot_store_ecap_z"].records


# Storage
cap0_store_z = (
    cap0_store_z
    .assign(lt = "FX")
    .set_index(["z","s","lt"])
    [["level"]]
)

ecap0_store_z = (
    ecap0_store_z
    .assign(lt = "FX")
    .set_index(["z","s","lt"])
    [["level"]]
)

In [ ]:
# Load commissioning and decommissioning plans
store_comm_decomm_cap_full = pd.read_csv(store_comm_decomm_path)

store = list(m["s"].records.uni)

# Storage
store_comm_decomm_cap = (
    store_comm_decomm_cap_full
    .query("Year<=@model_year")
    .query("Year>@prev_year")
    .drop(columns=["Year"])
)

store_commissioned_cap = (
    store_comm_decomm_cap
    .query("type=='commisioning'")
    .drop(columns=["type"])
)
store_decommissioned_cap = (
    store_comm_decomm_cap
    .query("type=='decommisioning'")
    .drop(columns=["type"])
)

# only s technologies
store_commissioned_cap_z = (
    store_commissioned_cap
    .query("Technology in @store")
    .drop(columns=["region"])
    .groupby(["Technology","zone"]).sum()
    .reset_index()
    .rename(columns={"Technology":"s","zone":"z","value":"level"})
    .assign(lt="LO")
    .set_index(["z","s","lt"])
    .div(1000)
)

store_decommissioned_cap_z = (
    store_decommissioned_cap
    .query("Technology in @store")
    .drop(columns=["region"])
    .groupby(["Technology","zone"]).sum()
    .reset_index()
    .rename(columns={"Technology":"s","zone":"z","value":"level"})
    .assign(lt="FX")
    .set_index(["z","s","lt"])
    .div(1000)
)




In [ ]:
cap_store_z = (
    cap0_store_z
    .sub(store_decommissioned_cap_z, fill_value=0)
    # .add(store_commissioned_cap_z, fill_value=0)
    .reset_index()
)

ecap_store_z = (
    ecap0_store_z
    .reset_index()
)


In [ ]:
cap_store_z_agg = pd.DataFrame(cap_store_z["z"].astype(str) +"."+ cap_store_z["s"].astype(str) +"."+ cap_store_z["lt"].astype(str),columns=["store_exist_pcap_z"])
cap_store_z_agg["level"] = cap_store_z[["level"]]

ecap_store_z_agg = pd.DataFrame(ecap_store_z["z"].astype(str) +"."+ ecap_store_z["s"].astype(str) +"."+ ecap_store_z["lt"].astype(str),columns=["store_exist_ecap_z"])
ecap_store_z_agg["level"] = ecap_store_z[["level"]]


In [ ]:
## Commissioned

# comm_store_cap_z = (
#     store_commissioned_cap_z
#     .reset_index()
# )

# comm_store_z_agg = pd.DataFrame(comm_store_cap_z["z"].astype(str) +"."+ comm_store_cap_z["s"].astype(str) +"."+ comm_store_cap_z["lt"].astype(str),columns=["gen_comm_pcap_z"])
# comm_store_z_agg["level"] = comm_store_cap_z[["level"]]


# comm_store_z_out = pd.DataFrame(wrapdd(comm_store_z_agg.values,comm_store_z_agg.columns[0],"parameter"))
# comm_store_z_out["out"] = comm_store_z_out[0]+" "+comm_store_z_out[1].astype(str)


In [ ]:
## Add existing cap to store dd file

store_dd = pd.read_csv(store_dd_path,skip_blank_lines=False,header=None)

cap_store_z_out = pd.DataFrame(wrapdd(cap_store_z_agg.values,cap_store_z_agg.columns[0],"parameter"))
cap_store_z_out["out"] = cap_store_z_out[0]+" "+cap_store_z_out[1].astype(str)

ecap_store_z_out = pd.DataFrame(wrapdd(ecap_store_z_agg.values,ecap_store_z_agg.columns[0],"parameter"))
ecap_store_z_out["out"] = ecap_store_z_out[0]+" "+ecap_store_z_out[1].astype(str)

store_dd_out = np.concatenate(
        (store_dd.values,
        ecap_store_z_out.loc[:,["out"]].values,
        cap_store_z_out.loc[:,["out"]].values,
        ),
        axis=0)

np.savetxt(
            store_dd_path, store_dd_out, delimiter=" ", fmt="%s"
        )

